### Luce–Tonina Δz solver

- Reads: daily harmonic regression output: revised_daily_temperature_harmonics_and_delta.csv
- Computes:lnA_ratio_LT; dphi_rad_LT; eta_LT; delta_z_m (Luce–Tonina Δz); delta_z_change_m
- Saves: daily_delta_z_from_LT.csv

In [1]:

# ============================================
# Luce–Tonina Δz solver from daily harmonics
# ============================================
import numpy as np
import pandas as pd
from pathlib import Path


# ============================================
# Helper: phase wrapping to [-π, π]
# ============================================

def wrap_phase(dphi):
    """Wrap phase difference to [-π, π]."""
    return (dphi + np.pi) % (2*np.pi) - np.pi


# ============================================
# Core Solver
# ============================================ 

def compute_delta_z_LuceTonina(
    df,
    ke=0.8e-6,
    period_s=86400.0,
    min_amp=1e-9,
    min_dphi=1e-9,
    clip_eta=30.0,
    smooth_days=0
):
    """Compute Luce–Tonina Δz from daily harmonic amplitudes and phases."""

    # Ensure amplitude positivity
    Aw = np.clip(df["Aw"].values, min_amp, None)
    As = np.clip(df["As"].values, min_amp, None)

    # Phase difference
    dphi = wrap_phase(df["phi_s_rad"] - df["phi_w_rad"])
    dphi = np.where(np.abs(dphi) < min_dphi, np.nan, dphi)

    # Log amplitude ratio
    lnA = np.log(Aw / As)

    # Tonina eta
    with np.errstate(divide="ignore", invalid="ignore"):
        eta = -(np.log(As / Aw)) / dphi
    eta[~np.isfinite(eta)] = np.nan
    eta = np.clip(eta, -clip_eta, clip_eta)

    # Luce–Tonina Δz
    omega = 2*np.pi / period_s
    scale = np.sqrt(ke / omega)

    with np.errstate(divide="ignore", invalid="ignore"):
        delta_z = dphi * scale * (eta + 1.0/eta)
    delta_z[~np.isfinite(delta_z)] = np.nan

    out = df.copy()
    out["lnA_ratio_LT"] = lnA
    out["dphi_rad_LT"] = dphi
    out["eta_LT"] = eta
    out["delta_z_m"] = delta_z

    # Optional smoothing
    if smooth_days > 1:
        out["delta_z_m_smooth"] = (
            out["delta_z_m"]
            .rolling(window=smooth_days, center=True, min_periods=1)
            .mean()
        )

    dz_col = "delta_z_m_smooth" if smooth_days > 1 else "delta_z_m"
    out["delta_z_change_m"] = out[dz_col].diff()

    return out


def main():
    in_csv  = "revised_daily_temperature_harmonics_and_delta.csv"
    out_csv = "daily_delta_z_from_LT.csv"

    df = pd.read_csv(in_csv, parse_dates=["date"]).set_index("date").sort_index()
    out = compute_delta_z_LuceTonina(df)

    out.to_csv(out_csv)
    print(f"Saved: {out_csv}")
    print(out[["delta_z_m"]].describe())

if __name__ == "__main__":
    main()

Saved: daily_delta_z_from_LT.csv
        delta_z_m
count  308.000000
mean     0.517809
std      8.132343
min     -3.602685
25%      0.026241
50%      0.060273
75%      0.100729
max    142.716048


C:\Users\Chinedu\AppData\Local\Temp\ipykernel_28384\2706596937.py:82: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df = pd.read_csv(in_csv, parse_dates=["date"]).set_index("date").sort_index()
